# H&M GraphSAGE seed49 — 기존 학습 재개, Drive 5epoch 저장
학습 source·seed49·300epoch·입력·평가·출력 경로는 기존과 같습니다. 로컬 최신/직전은 매epoch, Drive 최신/직전은5epoch 및 최종epoch마다 보존합니다. 런타임 소실 시 마지막 Drive 저장 뒤 미저장 epoch는 재계산합니다. 완료 결과는 재사용합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '214e5d1a64d3a9a5a4c790323ae71dab8851c608'
REPO = Path('/content/clv-hm-graphsage-v2-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 복구된 학교 Drive 입력 검증 — 학습 없음
원본 거래·상품과 M2/M4 참조 ZIP의 SHA256을 검사합니다. 기존 5조각 ZIP을 재설치하지 않습니다. 전체 전처리 후 M2 특징·M3 그래프·M4 가중치도 기존 실험과 대조합니다.


In [ ]:
# 읽기 전용 재개 점검: 학습/삭제 없음
import os, torch, json
from dataclasses import asdict
from pathlib import Path
print('PID',os.getpid(),'run',prepared['run_dir'],flush=True)
for label in screen.ORDER:
    row=screen.completed(prepared,label)
    print(label,'완료' if row is not None else '미완료',flush=True)
resume=Path(prepared['run_dir'])/'progress'/'resume'
for p in sorted(resume.glob('*_latest.pt')):
    print('파일',p.name,p.stat().st_size,flush=True)
    if 'graphsage_m4_lastweek' in p.name:
        ck=torch.load(p,map_location='cpu',weights_only=False)
        assert ck['identity']==asdict(screen.identity(prepared,'M4'))
        assert len(ck['history'])==ck['epoch']
        assert all(k in ck for k in ['model_state','optimizer_state','numpy_rng_state','torch_rng_state'])
        print('M4 검증 완료: 저장',ck['epoch'],'다음',ck['epoch']+1,flush=True)
        del ck
print('helper',Path('/content/clv_drive_checkpoint_retention_patch.py').exists(),flush=True)
print('Drive',drive_api.about().get(fields='storageQuota').execute(),flush=True)


In [ ]:
import hashlib
DATA = Path('/content/drive/MyDrive/논문/data')
EXPECTED_FILES = {
    'raw/hm/transactions_train.parquet': '0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b',
    'raw/hm/articles.csv': '25f21882053517f7684ea43f524225a9481a7bb9aab9cf0abfc7fe8c4576a72b',
    'references/clv_m2_adaptive_hm_seed49_lastweek_results.zip': '6d5159a888319b8a443a6907922a6e36fe9954c6b717dc2b1a6ff547c1d52bed',
    'references/clv_hm_m4c_lastweek_seed49_results.zip': '11fbfbddd21f966357cc872d43891bd078741b83037e88ee39a5558498ce40fe',
}
for name, expected in EXPECTED_FILES.items():
    path = DATA / name
    assert path.is_file(), f'학교 Drive 입력 파일 없음: {path}'
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
            digest.update(block)
    assert digest.hexdigest() == expected, f'원본과 다른 입력: {name}. 이전 ZIP을 재설치하지 마세요.'
print('학교 Drive 원본 입력·참조 4개 SHA256 일치')

import clv_hm_graphsage_final8_seed49 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이전 모듈이 메모리에 남아 있습니다. 이 런타임만 다시 시작한 후 위에서부터 실행하세요.'
# Fixed raw bytes are read from the runtime disk, not repeatedly from Drive.
import shutil
LOCAL_INPUT = Path('/content/hm_graphsage_inputs')
LOCAL_INPUT.mkdir(exist_ok=True)
for key, name in [('tx_path', 'transactions_train.parquet'), ('item_meta_path', 'articles.csv')]:
    source = DATA / 'raw/hm' / name
    target = LOCAL_INPUT / name
    expected = EXPECTED_FILES['raw/hm/' + name]
    def matches(path):
        if not path.is_file():
            return False
        h = hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(block)
        return h.hexdigest() == expected
    if not matches(target):
        temporary = target.with_suffix(target.suffix + '.partial')
        shutil.copyfile(source, temporary)
        assert matches(temporary), f'로컬 복사 해시 불일치: {name}'
        temporary.replace(target)
    screen.base.v3.SCHEMA['hm'][key] = str(target)
print('원본과 동일한 Colab 로컬 입력 준비 완료')
cfg = screen.configure(
    m2_reference=str(DATA / 'references/clv_m2_adaptive_hm_seed49_lastweek_results.zip'),
    m4_reference=str(DATA / 'references/clv_hm_m4c_lastweek_seed49_results.zip'),
    out_dir=str(DATA / 'results_v3_hm_graphsage_final8_seed49_school_v2'),
)
print('출력:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('검증 완료:', list(screen.ORDER), '/ seed49 / 각300epoch')
print('진행·checkpoint·결과 저장:', prepared['run_dir'])


## 기존 학습 재개
보존 설치 후 학습 셀을 실행합니다. 중간 로컬 저장은 Drive 업로드를 생략합니다. 예전 학습 로그의 “Drive 저장완료” 시간 표기는 저장 함수 전체 시간이며, 실제 Drive 업로드는 [저장 주기] 메시지로 구분합니다.


In [ ]:
# 최신287epoch 복구 및 저장 보존 정책 — 학습 조건 변경 없음
from google.colab import auth
auth.authenticate_user()
import google.auth,hashlib,urllib.request,importlib.util
from googleapiclient.discovery import build
credentials,_=google.auth.default()
drive_api=build('drive','v3',credentials=credentials,cache_discovery=False)
assert drive_api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']=='jungunb@u.sogang.ac.kr'
print('학교 Drive 인증 완료. 복구/보존 설치 전에는 학습하지 않습니다.',flush=True)

In [ ]:
# 과거 M1/M2 사본 정리는 완료되었습니다. 재개 시 삭제를 반복하지 않습니다.
print('과거 일회성 파일 정리 완료: 추가 삭제 없음. 아래 재개/보존 셀을 실행하세요.')


In [ ]:
# 보존 설치: 완료 M4의 Adam 상태는 요구하지 않습니다.
import hashlib, urllib.request, importlib.util
from pathlib import Path
assert cfg.epochs == 300 and cfg.seeds == (49,)
assert set(prepared['data']['splits']) == {'test'}
code=urllib.request.urlopen('https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/62a19e044753b8259674903c7cd29c580798c535/clv_drive_checkpoint_retention.py',timeout=60).read()
assert hashlib.sha256(code).hexdigest()=='32a69a7ba5e3f3d7c0f9ca3b0e0731d3075f8b547069afa4df3308f0bf873e98'
helper=Path('/content/clv_drive_checkpoint_retention_patch.py');helper.write_bytes(code)
spec=importlib.util.spec_from_file_location('clv_drive_checkpoint_retention_patch',helper)
retention=importlib.util.module_from_spec(spec);spec.loader.exec_module(retention)
retention.install(screen,drive_api,prepared)


In [ ]:
# 로컬 매epoch / Drive 5epoch + 최종epoch. 실험 source와 출력 경로 유지.
assert 'prepared' in globals()
assert getattr(screen.SafeProgressStore, '_retention_installed', False)
import hashlib, urllib.request, types
url='https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/6bb986df686bb52eacdf09f82960cce829a804c2/clv_hm_checkpoint_cadence.py'
with urllib.request.urlopen(url,timeout=60) as response: source=response.read()
assert hashlib.sha256(source).hexdigest()=='91ec25d8bae5d7eb685f8236c497aca10d26d69cca1c8ee52e8fec555c78e8f2'
cadence=types.ModuleType('verified_hm_cadence')
exec(compile(source,'<verified-hm-cadence>','exec'),cadence.__dict__)
cadence.install(screen,prepared)
result=screen.run(prepared)
print(json.dumps(result['reading'],ensure_ascii=False,indent=2))


## 시간 기록 확인 — 학습·평가 없음
아래 셀은 저장된 시간 CSV만 읽습니다. 학습 셀이 끝났거나 중단된 뒤 실행하세요. 마지막10개 epoch를 보여주며 전체 기록은 결과 ZIP의 `timing/`에 포함됩니다. 입력 준비 총 시간은 `preparation_timing.json`에 있습니다. 표본 시간은 첫3개 epoch만 있으며, 나머지 빈칸은 미측정입니다.

In [ ]:
import pandas as pd
from IPython.display import display
timing_files = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_files:
    timing = pd.concat([pd.read_csv(p) for p in timing_files], ignore_index=True)
    columns = ['model_id', 'epoch', 'train_seconds', 'save_seconds',
               'sample_input_seconds_per_batch', 'sample_compute_seconds_per_batch',
               'checkpoint_local_seconds', 'checkpoint_drive_seconds', 'checkpoint_metadata_seconds']
    display(timing[columns].tail(10))
    print('단위: 초. 표본 계측은 추가 동기화 비용을 포함합니다. test 성능을 측정하지 않습니다.')
else:
    print('아직 완료 epoch의 시간 기록이 없습니다. 초기 배치 시간은 학습 출력에서 확인하세요.')

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 GraphSAGE M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('GraphSAGE M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed49 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
import shutil
output = screen.export_zip(prepared, '/content/hm_graphsage_final8_seed49_results.zip')
src = Path(output)
backup = prepared['run_dir'] / src.name
shutil.copy2(src, backup)
assert backup.stat().st_size == src.stat().st_size
print('Drive 보관:', backup, '| ZIP 크기(MB):', round(src.stat().st_size / 1024**2, 3))
print('다운로드가 멈추면 위 Drive 경로에서 직접 다운로드하세요. checkpoint는 Drive에 유지됩니다.')
files.download(str(src))